# Batch lifetime analysis aligned to lick bouts

This notebook reads mouse and session dates from an Excel workbook, discovers all matching Doric and NI-DAQ runs, detects lick bouts with one fixed rule, and aligns vendor `Tau01` to bout onset. Bouts can be selected regardless of Ensure, only when associated with Ensure, or only when no Ensure is nearby. Results are averaged across bouts within each mouse before calculating the across-mouse mean.

In [ ]:
from datetime import date, datetime
from pathlib import Path
import csv
import matplotlib.pyplot as plt
import numpy as np
from openpyxl import load_workbook
from fluopulse_analysis import (
    extract_perievent,
    find_doric_files,
    find_lick_bouts,
    infer_session_identity,
    nidaq_paths_for_doric,
    process_aligned_session,
)

## Configuration

Set `BOUT_MODE` to `"all"`, `"with_ensure"`, or `"without_ensure"`. A bout is Ensure-associated when an Ensure onset falls between 5 seconds before and 1 second after the first lick. Keep these definitions fixed across experimental groups.

In [ ]:
SESSION_TABLE = Path.home() / "Documents" / "SC_FLiPmice_2025.xlsx"
DORIC_ROOT = Path(r"Z:\FLIM FLIP")
PHOTOMETRY_ROOT = Path(r"Z:\Photometry")
OUTPUT_FOLDER = Path.cwd() / "analysis_output_batch_lick_bouts"
BOUT_MODE = "all"  # all, with_ensure, or without_ensure
MAX_INTERLICK_GAP_S = 1.0
MIN_LICKS_PER_BOUT = 5
ENSURE_WINDOW_S = (-5.0, 1.0)
PERIEVENT_WINDOW_S = (-20.0, 60.0)
BASELINE_WINDOW_S = (-10.0, 0.0)
SAMPLE_INTERVAL_S = 0.1
MIN_NIDAQ_COVERAGE = 0.99
if BOUT_MODE not in {"all", "with_ensure", "without_ensure"}:
    raise ValueError(f"Unknown BOUT_MODE: {BOUT_MODE}")

## Read the mouse/date table

In [ ]:
def normalize_date(value):
    if isinstance(value, (date, datetime)):
        return value.strftime("%y%m%d")
    return str(int(value)).zfill(6)

workbook = load_workbook(SESSION_TABLE, data_only=True, read_only=True)
worksheet = workbook.active
rows = list(worksheet.iter_rows(values_only=True))
workbook.close()
headers = [str(value).strip() if value is not None else "" for value in rows[0]]
if not headers or headers[0].casefold() != "mouse":
    raise ValueError("The first workbook column must be named 'mouse'")
session_records = []
for row in rows[1:]:
    if not row or row[0] is None:
        continue
    mouse = str(row[0]).strip()
    for column, value in zip(headers[1:], row[1:], strict=False):
        if value is not None:
            session_records.append({"mouse": mouse, "date": normalize_date(value), "source_column": column})
print(f"Loaded {len(session_records)} mouse-date sessions")
session_records

## Process every run

Missing or incomplete sessions are recorded in `qc_rows` rather than stopping the whole batch. NI-DAQ coverage below the configured threshold is excluded from peri-event analysis.

In [ ]:
qc_rows = []
bout_rows = []
aligned_trials = []
trial_mice = []
trial_has_ensure = []
relative_time = None
for record in session_records:
    mouse = record["mouse"]
    session_date = record["date"]
    doric_files = find_doric_files(mouse, session_date, doric_root=DORIC_ROOT)
    if not doric_files:
        qc_rows.append({"mouse": mouse, "date": session_date, "run": "", "status": "missing Doric recording"})
        continue
    for doric_file in doric_files:
        identity = infer_session_identity(doric_file)
        paths = nidaq_paths_for_doric(doric_file, photometry_root=PHOTOMETRY_ROOT)
        if not paths.nidaq.is_file():
            qc_rows.append({"mouse": mouse, "date": session_date, "run": identity.run, "status": "missing NI-DAQ recording"})
            continue
        try:
            session = process_aligned_session(
                doric_file,
                paths.nidaq,
                running_path=paths.running if paths.running.is_file() else None,
            )
            bouts = find_lick_bouts(
                session.doric_licks.onset_times,
                max_interlick_gap_seconds=MAX_INTERLICK_GAP_S,
                min_licks=MIN_LICKS_PER_BOUT,
            )
            ensure_times = np.asarray(session.ensure_pulses.onset_times)
            has_ensure = np.array([
                np.any(
                    (ensure_times >= onset + ENSURE_WINDOW_S[0])
                    & (ensure_times <= onset + ENSURE_WINDOW_S[1])
                )
                for onset in bouts.onset_times
            ], dtype=bool)
            if BOUT_MODE == "with_ensure":
                selected = has_ensure
            elif BOUT_MODE == "without_ensure":
                selected = ~has_ensure
            else:
                selected = np.ones(bouts.onset_times.size, dtype=bool)
            status = "ok" if session.nidaq_coverage_fraction >= MIN_NIDAQ_COVERAGE else "incomplete NI-DAQ coverage"
            qc_rows.append({
                "mouse": mouse,
                "date": session_date,
                "run": identity.run,
                "status": status,
                "coverage_percent": 100 * session.nidaq_coverage_fraction,
                "sync_rms_ms": 1000 * session.alignment.rms_residual_seconds,
                "clock_drift_ppm": session.alignment.drift_ppm,
                "lick_count": session.doric_licks.onset_times.size,
                "bout_count": bouts.onset_times.size,
                "selected_bout_count": int(selected.sum()),
                "ensure_count": ensure_times.size,
            })
            if status != "ok" or not np.any(selected):
                continue
            selected_indices = np.flatnonzero(selected)
            relative_time, trials = extract_perievent(
                session.lifetime_time_nidaq,
                session.doric.tau_ns,
                bouts.onset_times[selected],
                window_seconds=PERIEVENT_WINDOW_S,
                sample_interval_seconds=SAMPLE_INTERVAL_S,
            )
            baseline_mask = (relative_time >= BASELINE_WINDOW_S[0]) & (relative_time < BASELINE_WINDOW_S[1])
            baseline = np.nanmean(trials[:, baseline_mask], axis=1, keepdims=True)
            trials = trials - baseline
            for local_index, bout_index in enumerate(selected_indices):
                ensure_candidates = ensure_times[
                    (ensure_times >= bouts.onset_times[bout_index] + ENSURE_WINDOW_S[0])
                    & (ensure_times <= bouts.onset_times[bout_index] + ENSURE_WINDOW_S[1])
                ]
                ensure_latency = float(ensure_candidates[-1] - bouts.onset_times[bout_index]) if ensure_candidates.size else np.nan
                bout_rows.append({
                    "mouse": mouse,
                    "date": session_date,
                    "run": identity.run,
                    "bout_index": int(bout_index),
                    "bout_onset_s": bouts.onset_times[bout_index],
                    "bout_end_s": bouts.offset_times[bout_index],
                    "lick_count": int(bouts.lick_counts[bout_index]),
                    "has_ensure": bool(has_ensure[bout_index]),
                    "ensure_latency_s": ensure_latency,
                })
                aligned_trials.append(trials[local_index])
                trial_mice.append(mouse)
                trial_has_ensure.append(bool(has_ensure[bout_index]))
        except Exception as error:
            qc_rows.append({"mouse": mouse, "date": session_date, "run": identity.run, "status": f"error: {error}"})
print(f"Processed {len(qc_rows)} run records")
print(f"Selected {len(aligned_trials)} lick bouts")
qc_rows

## Plot bout-aligned lifetime

Thin lines are per-mouse means. The bold line and shaded region summarize mouse means, so mice with more lick bouts do not automatically receive more weight.

In [ ]:
if not aligned_trials:
    raise ValueError("No bouts passed the configured selection and QC criteria")
trials = np.vstack(aligned_trials)
trial_mice = np.asarray(trial_mice)
trial_has_ensure = np.asarray(trial_has_ensure, dtype=bool)
mouse_names = np.unique(trial_mice)
mouse_means = np.vstack([np.nanmean(trials[trial_mice == mouse], axis=0) for mouse in mouse_names])
grand_mean = np.nanmean(mouse_means, axis=0)
valid_mice = np.sum(np.isfinite(mouse_means), axis=0)
grand_sem = np.nanstd(mouse_means, axis=0, ddof=1) / np.sqrt(np.maximum(valid_mice, 1))
figure, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True, constrained_layout=True)
order = np.argsort(trial_has_ensure.astype(int))[::-1]
image = axes[0].imshow(
    trials[order],
    aspect="auto",
    origin="lower",
    extent=[relative_time[0], relative_time[-1], 0, trials.shape[0]],
    cmap="coolwarm",
)
axes[0].axvline(0, color="black", linestyle="--")
axes[0].set_ylabel("Selected bout")
figure.colorbar(image, ax=axes[0], label="Delta tau (ns)")
for mouse, trace in zip(mouse_names, mouse_means, strict=True):
    axes[1].plot(relative_time, trace, color="0.75", linewidth=1)
axes[1].plot(relative_time, grand_mean, color="tab:blue", linewidth=2.5, label=f"Mouse mean (n={mouse_names.size})")
axes[1].fill_between(relative_time, grand_mean - grand_sem, grand_mean + grand_sem, color="tab:blue", alpha=0.2)
axes[1].axvline(0, color="tab:orange", linestyle="--")
axes[1].set(xlabel="Time from lick-bout onset (s)", ylabel="Delta tau (ns)")
axes[1].legend()
plt.show()

## Export results

In [ ]:
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
def write_dict_rows(path, rows):
    if not rows:
        return
    fieldnames = []
    for row in rows:
        for key in row:
            if key not in fieldnames:
                fieldnames.append(key)
    with path.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
write_dict_rows(OUTPUT_FOLDER / "session_qc.csv", qc_rows)
write_dict_rows(OUTPUT_FOLDER / "bout_metadata.csv", bout_rows)
np.savez_compressed(
    OUTPUT_FOLDER / "bout_aligned_lifetime.npz",
    relative_time_s=relative_time,
    delta_tau_ns=trials,
    mouse=trial_mice,
    has_ensure=trial_has_ensure,
)
figure.savefig(OUTPUT_FOLDER / "bout_aligned_lifetime.png", dpi=200)
print(f"Wrote results to {OUTPUT_FOLDER.resolve()}")

Review `session_qc.csv` before interpreting the aggregate. Missing files, incomplete NI-DAQ coverage, poor clock fits, or sessions with no qualifying bouts remain visible rather than being silently treated as zero responses.